# Getting started: Kaggle Top Solutions and Strategies

Three record types: `competition`, `solution` (a public writeup of any rank with link and summary) and `strategy` (one technique in one solution). They come split into competitions.csv, solutions.csv and strategies.csv, and merged in `kaggle_all_solutions_and_strategies_merged.csv` (the main file, filtered by `record_type`). This notebook loads the split files and asks typical questions.

In [ ]:
import glob, json, os
import pandas as pd

# Kaggle input, or the data/ folder of the GitHub repository
DATA = os.path.dirname((glob.glob('/kaggle/input/**/competitions.csv', recursive=True)
                        + glob.glob('../data/competitions.csv') + glob.glob('data/competitions.csv'))[0])
comps = pd.read_csv(os.path.join(DATA, 'competitions.csv'))
sols = pd.read_csv(os.path.join(DATA, 'solutions.csv'))
strat = pd.read_csv(os.path.join(DATA, 'strategies.csv'))
# merged main file: pd.read_csv(os.path.join(DATA, 'kaggle_all_solutions_and_strategies_merged.csv'), low_memory=False)
pd.Series({'competitions': len(comps), 'solutions': len(sols), 'strategies': len(strat)})

## Charts (interactive, Plotly)

Hover over any mark for exact values.

In [ ]:
import os, uuid
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.offline import get_plotlyjs_version

PLOTLY_JS = f'https://cdn.plot.ly/plotly-{get_plotlyjs_version()}.min'

def show(fig):
    """The saved-notebook viewer has require.js, which turns plotly's CDN script into an AMD module (no global
    Plotly). There, load plotly.js once as a global with define hidden, and draw every figure from that."""
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') != 'Batch':
        return fig.show()
    div = 'fig-' + uuid.uuid4().hex[:8]
    display(HTML(f'''<div id="{div}" style="height:{fig.layout.height or 450}px"></div>
<script>(function () {{
  var spec = {fig.to_json()};
  if (!window.__plotlyGlobal) window.__plotlyGlobal = new Promise(function (resolve) {{
    if (window.Plotly) return resolve(window.Plotly);
    var saved = window.define; window.define = undefined;
    var s = document.createElement('script'); s.src = '{PLOTLY_JS}.js';
    s.onload = function () {{ window.define = saved; resolve(window.Plotly); }};
    s.onerror = function () {{ window.define = saved; }};
    document.head.appendChild(s);
  }});
  window.__plotlyGlobal.then(function (Plotly) {{
    Plotly.newPlot('{div}', spec.data, spec.layout, {{responsive: true, displaylogo: false}});
  }});
}})();</script>'''))

INK, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#e7e6e2', '#fcfcfb'
BLUE, ORANGE = '#2a78d6', '#eb6834'
RANK_COLORS = {'1-10': '#184f95', '11-100': '#2a78d6', '101+': '#86b6ef', 'unknown': '#c3c2b7'}
BLUES = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b']

def style(fig, title, subtitle='', height=440, grid=True):
    fig.update_layout(
        title=dict(text=f'<b>{title}</b><br><span style="font-size:13px;color:{MUTED}">{subtitle}</span>', x=0.01),
        template='plotly_white', height=height, paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        font=dict(family='Inter, Segoe UI, Helvetica, Arial, sans-serif', size=13, color=INK),
        margin=dict(l=10, r=20, t=110, b=10), hoverlabel=dict(bgcolor='white', font_size=12),
        legend=dict(orientation='h', yanchor='bottom', y=1.0, x=0, title_text=''))
    for axis in (fig.update_xaxes, fig.update_yaxes):
        axis(showgrid=grid, gridcolor=GRID, zeroline=False, linecolor=GRID, tickfont_color=MUTED, title_font_color=MUTED)
    try:
        fig.update_layout(barcornerradius=4)  # rounded bar ends, plotly >= 5.19
    except ValueError:
        pass
    return fig

In [ ]:
final = strat[(strat.usage_status == 'final_used') & ~strat.technique_id.str.startswith('new:')]
top15 = final.groupby('technique_name').writeup_id.nunique().nlargest(15).sort_values().reset_index(name='solutions')
fig = px.bar(top15, x='solutions', y='technique_name', orientation='h', color_discrete_sequence=[BLUE],
             labels={'technique_name': '', 'solutions': 'solutions using it in the final pipeline'})
show(style(fig, 'Most frequent techniques in final solutions', 'distinct solutions per technique (taxonomy concepts)', height=520))

In [ ]:
bucket = pd.cut(sols['rank'], [0, 10, 100, float('inf')], labels=['1-10', '11-100', '101+']).astype(str).replace('nan', 'unknown')
by_year = sols.assign(final_rank=bucket).groupby(['year', 'final_rank']).size().reset_index(name='writeups')
fig = px.bar(by_year, x='year', y='writeups', color='final_rank', color_discrete_map=RANK_COLORS,
             category_orders={'final_rank': list(RANK_COLORS)}, labels={'year': '', 'final_rank': 'final rank'})
fig.update_traces(marker_line_color=SURFACE, marker_line_width=1.5)
fig.update_layout(hovermode='x unified', bargap=0.3)
show(style(fig, 'Solution writeups per year, by final rank', 'every rank is collected, not only the top'))

In [ ]:
t10 = final[final['rank'] <= 10].assign(modality=lambda d: d.modalities.fillna('[]').map(json.loads)).explode('modality')
names = t10.groupby('technique_name').writeup_id.nunique().nlargest(12).index
mods = t10.groupby('modality').writeup_id.nunique().nlargest(6).index
share = (t10[t10.technique_name.isin(names)].groupby(['modality', 'technique_name']).writeup_id.nunique()
         .div(t10.groupby('modality').writeup_id.nunique(), level='modality').unstack(fill_value=0).loc[mods, names])
fig = px.imshow(share, color_continuous_scale=BLUES, zmin=0, aspect='auto', labels=dict(x='', y='', color=''))
fig.update_traces(xgap=2, ygap=2, hovertemplate='%{y} · %{x}<br>%{z:.0%} of top-10 solutions<extra></extra>')
fig.update_coloraxes(colorbar=dict(tickformat='.0%', thickness=12))
show(style(fig, 'Top-10 solutions: technique use by data modality',
           'share of the top-10 solutions of each modality that used the technique', height=400, grid=False))

In [ ]:
used = strat[strat.usage_status == 'final_used']
rate = lambda part: part.groupby('technique_name').writeup_id.nunique() / part.writeup_id.nunique()
gap = pd.DataFrame({'top 10': rate(used[used['rank'] <= 10]), 'beyond 100': rate(used[used['rank'] > 100])}).fillna(0)
gap = gap.assign(diff=gap['top 10'] - gap['beyond 100']).nlargest(12, 'diff').sort_values('diff')
fig = go.Figure(go.Scatter(x=[v for a, b in zip(gap['beyond 100'], gap['top 10']) for v in (a, b, None)],
                           y=[v for n in gap.index for v in (n, n, None)], mode='lines',
                           line=dict(color=GRID, width=4), hoverinfo='skip', showlegend=False))
for col, color in (('top 10', BLUE), ('beyond 100', ORANGE)):
    fig.add_trace(go.Scatter(x=gap[col], y=gap.index, mode='markers', name=f'rank {col}',
                             marker=dict(size=12, color=color, line=dict(color=SURFACE, width=2)),
                             hovertemplate='%{y}<br>' + f'rank {col}: ' + '%{x:.0%} of solutions<extra></extra>'))
fig.update_xaxes(tickformat='.0%', title_text='share of solutions using it in the final pipeline')
show(style(fig, 'What separates top-10 solutions from those ranked beyond 100',
           'the 12 techniques with the largest gap in use', height=500))

## Which techniques did top-10 solutions use, by data modality?

In [ ]:
top = strat[(strat['rank'] <= 10) & (strat.usage_status == 'final_used')].copy()
top['modality'] = top.modalities.fillna('[]').map(json.loads)
top = top.explode('modality')
(top.groupby('modality').technique_name.value_counts().groupby(level=0).head(5).to_frame('solutions'))

## All solutions of one competition, best rank first

In [ ]:
slug = 'child-mind-institute-detect-sleep-states'
sols[sols.competition_slug == slug].sort_values('rank')[['rank', 'rank_status', 'solution_title', 'writeup_url', 'solution_summary']]

## Top-10 versus beyond-100 solutions: which final techniques differ most?

In [ ]:
final = strat[strat.usage_status == 'final_used']
share = lambda part: part.groupby('technique_name').writeup_id.nunique() / part.writeup_id.nunique()
cmp = pd.DataFrame({'top10': share(final[final['rank'] <= 10]), 'beyond100': share(final[final['rank'] > 100])}).fillna(0)
cmp.assign(diff=cmp.top10 - cmp.beyond100).sort_values('diff', ascending=False).head(10)

## Find competitions like your problem (simple keyword match)

In [ ]:
def similar(text, n=5):
    words = set(text.lower().split())
    hay = (comps.competition_title.fillna('') + ' ' + comps.target_summary.fillna('')).str.lower()
    score = hay.map(lambda s: len(words & set(s.split())))
    return comps.assign(score=score).sort_values('score', ascending=False)[['competition_slug', 'year', 'metric_name', 'target_summary']].head(n)

similar('predict sleep events from wrist accelerometer time series')

## Tools for an AI agent

These functions are small enough to register as tools for an LLM agent (for example as Python functions in a tool-calling framework). They return counts and a source link for each technique, so the agent can cite `writeup_url` instead of inventing results.

In [ ]:
def strategies_for(task_type=None, modality=None, max_rank=10,
                   status="final_used", top=15):
    """Most common techniques in matching solutions, one link each."""
    s = strat[strat.usage_status == status]
    if task_type:
        s = s[s.task_type.str.contains(f'"{task_type}"', na=False)]
    if modality:
        s = s[s.modalities.str.contains(f'"{modality}"', na=False)]
    if max_rank:
        s = s[s["rank"] <= max_rank]
    out = s.groupby("technique_name").agg(  # each strategy row has its link
        solutions=("writeup_id", "nunique"), example=("writeup_url", "first"))
    return out.sort_values("solutions", ascending=False).head(top)

strategies_for('segmentation', 'image')

In [ ]:
def advise(problem, n_competitions=5, max_rank=20, per_stage=3):
    """Similar competitions -> their solutions up to max_rank -> final techniques by stage, plus dropped ideas."""
    found = similar(problem, n_competitions)
    s = strat[strat.competition_slug.isin(found.competition_slug) & (strat['rank'] <= max_rank)]
    plan = (s[s.usage_status == 'final_used'].groupby(['pipeline_stage', 'technique_name'])
            .agg(solutions=('writeup_id', 'nunique'), example=('writeup_url', 'first'))
            .sort_values('solutions', ascending=False).groupby(level=0).head(per_stage).sort_index(level=0, sort_remaining=False))
    dropped = (s[s.usage_status.isin(['tested_rejected', 'failed'])].groupby('technique_name').writeup_id.nunique()
               .sort_values(ascending=False).head(8).rename('solutions'))
    return found[['competition_slug', 'year', 'metric_name']], plan, dropped

found, plan, dropped = advise('predict sleep events from wrist accelerometer time series')
display(found, plan)
dropped

## Ideas that top teams tried and dropped

In [ ]:
strat[strat.usage_status.isin(['tested_rejected', 'failed']) & (strat['rank'] <= 10)][['competition_slug', 'rank', 'technique_name', 'technique_summary']].head(15)

## Frameworks named in top-10 solutions

In [ ]:
fw = sols[sols['rank'] <= 10].frameworks_libraries.dropna().map(json.loads).explode()
fw.value_counts().head(15)